# Week 10 — IBKR Options Chain Snapshotting
Connect to IBKR paper trading, fetch live option chains, store as Parquet.

**Prerequisites:**
- TWS or IB Gateway running (paper trading, port 7497)
- API enabled: TWS → Edit → Global Configuration → API → Enable ActiveX and Socket Clients
- `pip install ib_async`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import logging
logging.basicConfig(level=logging.INFO)

from optlab_research.ibkr.snapshot import IBKRSnapshot, SnapshotConfig
from optlab_research.ibkr.store import ChainStore
import duckdb

print("IBKR module loaded")

## 1. Connect and Fetch SPY

In [ ]:
config = SnapshotConfig(
    port=7497,  # paper trading
    max_expirations=3,
    max_strikes_atm=10,
)
store = ChainStore("../../data/ibkr_chains")

with IBKRSnapshot(config) as snap:
    quotes = snap.fetch("SPY")
    df = snap.to_polars(quotes)

print(f"Fetched {len(quotes)} quotes")
print(df.head(10))

## 2. Write to Parquet and Query via DuckDB

In [ ]:
filepath = store.write("SPY", df)
print(f"Written: {filepath}")

con = duckdb.connect()
result = con.execute(f"""
    SELECT
        right,
        expiration,
        COUNT(*) as n_contracts,
        AVG(implied_vol) as avg_iv,
        AVG(bid) as avg_bid,
        AVG(ask) as avg_ask
    FROM read_parquet('{filepath}')
    GROUP BY right, expiration
    ORDER BY expiration, right
""").df()
print(result)

## 3. Volatility Surface

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import polars as pl

calls = df.filter(pl.col("right") == "C").drop_nulls(["strike", "implied_vol"])
if not calls.is_empty():
    by_expiry = calls.group_by("expiration").agg(pl.col("strike"), pl.col("implied_vol"))
    fig, ax = plt.subplots(figsize=(12, 5))
    for row in by_expiry.iter_rows(named=True):
        strikes = row["strike"]
        ivs = [iv * 100 for iv in row["implied_vol"]]
        ax.plot(strikes, ivs, marker='o', markersize=3, label=row["expiration"])
    ax.set_xlabel("Strike")
    ax.set_ylabel("Implied Volatility (%)")
    ax.set_title("SPY Vol Smile by Expiration")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("spy_vol_surface.png", dpi=150)
    print("Saved: spy_vol_surface.png")
else:
    print("No call data available — run during market hours")

## 4. Manifest Summary

In [ ]:
summary = store.summary()
for k, v in summary.items():
    print(f"{k}: {v}")

## Acceptance Criterion
`python scripts/snapshot_once.py` connects to IBKR paper, pulls SPY's full chain,
writes to Parquet, and the file is queryable from DuckDB.
Run the script from the repo root to verify end-to-end.